<a href="https://colab.research.google.com/github/supriya-006/FlyRank_Assignment/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/supriya-006/FlyRank_Assignment/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:
# Confirm dtypes for the two flag columns before trusting boolean logic on them
con.sql(f"""
DESCRIBE SELECT client_has_gsc, client_has_ga4
FROM '{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet' LIMIT 0
""").show()

features = con.sql(f"""
SELECT
    content_hash_id,
    report_date,
    AVG(sessions_paid) OVER w   AS sessions_paid_7d_avg,
    AVG(sessions_direct) OVER w AS sessions_direct_7d_avg,
    SUM(ai_gemini + ai_claude + ai_meta) OVER w AS ai_referral_7d_total,
    AVG(scroll_events) OVER w  AS scroll_events_7d_avg,
    (client_has_gsc AND client_has_ga4) AS has_full_tracking
FROM '{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet'
WINDOW w AS (
    PARTITION BY content_hash_id
    ORDER BY report_date
    RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW
)
""").df()

features.head()

**Five features, built from confirmed columns:**

1. `sessions_paid_7d_avg` — trailing 7-day average of `sessions_paid` per content item.
   Knowable at the decision moment because it only uses `report_date` values on or
   before that date.
2. `sessions_direct_7d_avg` — trailing 7-day average of `sessions_direct`, same window logic.
   Knowable at the decision moment because it's strictly backward-looking.
3. `ai_referral_7d_total` — trailing 7-day sum of `ai_gemini + ai_claude + ai_meta`.
   Knowable at the decision moment because AI-referral traffic on past days is
   already logged by the time a decision is made.
4. `scroll_events_7d_avg` — trailing 7-day average of `scroll_events`.
   Knowable at the decision moment because engagement events are recorded same-day
   or earlier, never in advance.
5. `has_full_tracking` — `client_has_gsc AND client_has_ga4` (static per client).
   Knowable at the decision moment because it's a client setup fact, not an outcome —
   true or false regardless of which day you ask.

Dtypes for `client_has_gsc`/`client_has_ga4` aren't confirmed yet (could be BOOLEAN or
0/1 INTEGER) — the code below checks both.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.